In [17]:
import pandas as pd
import os
import os.path as op
import glob
import re

# Find all .nii.gz files in habenula-rois directory
rois_dir = "/Users/chloehampson/Desktop/habenula-rois"
nii_files = glob.glob(op.join(rois_dir, "*.nii.gz"))

# Extract subject IDs from filenames
subject_ids = []
for nii_file in nii_files:
    filename = op.basename(nii_file)
    # Extract subject ID using regex (matches sub-XXXXX pattern)
    match = re.match(r'(sub-\d+)_', filename)
    if match:
        subject_ids.append(match.group(1))

# Create DataFrame and save to TSV
drawn_subjects = sorted(set(subject_ids))  # Remove duplicates and sort
drawn_df = pd.DataFrame({'subject': drawn_subjects})

output_path = op.join(rois_dir, "drawn-habenulas.tsv")
drawn_df.to_csv(output_path, sep='\t', index=False)

print(f"Found {len(drawn_subjects)} unique subjects with drawn habenulas")
print(f"Saved to: {output_path}")
print(f"\nFirst 10 subjects:")
print(drawn_subjects[:10])

# Load habenula-rois_subjects.tsv for comparison
rois_path = op.join(rois_dir, "habenula-rois_subjects.tsv")
rois_df = pd.read_csv(rois_path, sep='\t')
rois_subjects = rois_df['subject'].values

print(f"\nROIs subjects: {len(rois_subjects)} subjects")

Found 1482 unique subjects with drawn habenulas
Saved to: /Users/chloehampson/Desktop/habenula-rois/drawn-habenulas.tsv

First 10 subjects:
['sub-0050004', 'sub-0050005', 'sub-0050006', 'sub-0050007', 'sub-0050008', 'sub-0050010', 'sub-0050011', 'sub-0050012', 'sub-0050013', 'sub-0050014']

ROIs subjects: 1328 subjects


In [18]:
# Extract subjects from the original table file
table_path = op.join(rois_dir, "sub-group_task-rest_desc-1S2StTesthabenula_table.txt")

# Read the table file
with open(table_path, 'r') as f:
    lines = f.readlines()

# Find the header line and extract subjects
original_subjects = []
for line in lines:
    # Skip comment lines and empty lines
    if line.strip() and not line.startswith('#'):
        # Split by whitespace and get the first column (Subj)
        parts = line.split()
        if parts and parts[0] != 'Subj':  # Skip header row if present
            subj = parts[0]
            # Only add if it looks like a subject ID
            if subj.startswith('sub-'):
                original_subjects.append(subj)

# Remove duplicates and sort
original_subjects = sorted(set(original_subjects))

# Save to TSV
original_tsv_path = op.join(rois_dir, "original-subjects.tsv")
original_df = pd.DataFrame({'subject': original_subjects})
original_df.to_csv(original_tsv_path, sep='\t', index=False)

print(f"Found {len(original_subjects)} unique subjects in original table")
print(f"Saved to: {original_tsv_path}")
print(f"\nFirst 10 subjects:")
print(original_subjects[:10])

Found 1584 unique subjects in original table
Saved to: /Users/chloehampson/Desktop/habenula-rois/original-subjects.tsv

First 10 subjects:
['sub-0050004', 'sub-0050005', 'sub-0050006', 'sub-0050007', 'sub-0050008', 'sub-0050010', 'sub-0050011', 'sub-0050012', 'sub-0050013', 'sub-0050014']


In [19]:
# Compare the drawn subjects to original subjects
# Find subjects in drawn habenulas that are NOT in original
drawn_only = set(drawn_subjects) - set(original_subjects)
original_only = set(original_subjects) - set(drawn_subjects)

print(f"Subjects with drawn habenulas but NOT in original table: {len(drawn_only)}")
print(sorted(drawn_only))

print(f"\nSubjects in original table but NO drawn habenulas: {len(original_only)}")
print(sorted(original_only))

# Show overlap
overlap = set(drawn_subjects) & set(original_subjects)
print(f"\nSubjects in both: {len(overlap)}")

Subjects with drawn habenulas but NOT in original table: 0
[]

Subjects in original table but NO drawn habenulas: 102
['sub-0050282', 'sub-0050332', 'sub-0050345', 'sub-0050354', 'sub-0050360', 'sub-0050570', 'sub-0050694', 'sub-0050702', 'sub-0050818', 'sub-0051000', 'sub-0051008', 'sub-28693', 'sub-28799', 'sub-28974', 'sub-29104', 'sub-29443', 'sub-29617', 'sub-29628', 'sub-29867', 'sub-29868', 'sub-29869', 'sub-29870', 'sub-29872', 'sub-29873', 'sub-29874', 'sub-29875', 'sub-29876', 'sub-29877', 'sub-29878', 'sub-29879', 'sub-29881', 'sub-29882', 'sub-29883', 'sub-29885', 'sub-29886', 'sub-29888', 'sub-29891', 'sub-29892', 'sub-29894', 'sub-29895', 'sub-29896', 'sub-29897', 'sub-29899', 'sub-29900', 'sub-29901', 'sub-29904', 'sub-29905', 'sub-29906', 'sub-29907', 'sub-29908', 'sub-29911', 'sub-29912', 'sub-29913', 'sub-29915', 'sub-29916', 'sub-29917', 'sub-29997', 'sub-29998', 'sub-30000', 'sub-30001', 'sub-30004', 'sub-30005', 'sub-30006', 'sub-30007', 'sub-30008', 'sub-30009', '

In [20]:
# Show subjects with drawn habenulas not in original table
print(f"Subjects with drawn habenulas but NOT in original table ({len(drawn_only)} subjects):\n")
print(sorted(drawn_only))

Subjects with drawn habenulas but NOT in original table (0 subjects):

[]


In [21]:
# Save drawn_only subjects to CSV (subjects with drawings but not in original table)
output_csv_path = "/Users/chloehampson/Desktop/habenula-rois/drawn_not_in_original.csv"
drawn_only_df = pd.DataFrame({'subject': sorted(drawn_only)})
drawn_only_df.to_csv(output_csv_path, index=False)
print(f"Saved drawn-only subjects to: {output_csv_path}")

Saved drawn-only subjects to: /Users/chloehampson/Desktop/habenula-rois/drawn_not_in_original.csv
